# Current BERT, NRC v2.1, and MEmoLon: comparative VAD results

One workflow compares the **same three estimators** on (1) 20 synthetic English context pairs and (2) 50 saved news items across six chains, totaling 1,200 steps. No news rewrites or LLM evaluations are regenerated. Current BERT is evaluated using the same cached revision for both lexical comparisons.

**Normalization to 0–1:** BERT `(score - 1)/4`; NRC v2.1 `(native + 1)/2`; MEmoLon `(native - 1)/8`. Native scales are 1–5, −1–1, and 1–9. Both lexical scorers also export scores on the common 1–5 scale. Normalization matches nominal amplitudes, not calibration or accuracy.

Only unified results are exported. Earlier pilot notebooks and duplicate historical output copies have been removed; the initial NRC comparison remains accessible in Git at commit `d09cd838edbd827750933181775efe315c55763f`.

Before the first execution, cache the English lexicons with `uv run python scripts/download_vad_comparison_resources.py`. The comparison uses locally cached current BERT weights.

The final cell exports all interactive charts to `output/audit/SimulationVADModelComparison/comparison_dashboard.html`, including Plotly itself for offline viewing. This file retains the visual comparison when notebook outputs are cleared before a commit.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display
from plotly.subplots import make_subplots

from misinformation_simulation.analysis.three_model_vad_comparison import (
    MODEL_LABELS,
    run_three_model_context_comparison,
    run_three_model_simulation_comparison,
)
from misinformation_simulation.analysis.vad_model_comparison import DEFAULT_COMPARISON_CHAINS

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
MEMOLON_PATH = PROJECT_ROOT / ".cache/vad_lexicon_comparison/memolon_en.tsv"
NRC_PATH = PROJECT_ROOT / ".cache/nrc_vad/NRC-VAD-Lexicon-v2.1.zip"
BERT_CACHE_PATH = PROJECT_ROOT / ".cache/vad_lexicon_comparison/current_bert_saved_tests.json"
CONTEXT_INPUT = PROJECT_ROOT / "data/synthetic/vad_context_contrast_news.csv"
HISTORICAL_BASELINE = (
    PROJECT_ROOT / "output/audit/VADContextContrastAudit/vad_context_contrast_scored.csv"
)
RUNS_DIR = PROJECT_ROOT / "output/interaction_graph/app_runs"
OUTPUT_DIR = PROJECT_ROOT / "output/audit/SimulationVADModelComparison"
CONTEXT_OUTPUT_DIR = PROJECT_ROOT / "output/audit/VADThreeModelContextContrastAudit"
CHAIN_CODES = DEFAULT_COMPARISON_CHAINS
COLORS = {"Current BERT": "#2563eb", "NRC v2.1": "#e07a19", "MEmoLon MTL_grouped": "#16856b"}
FIGURES = {}
for path in (MEMOLON_PATH, NRC_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Cache the required lexicon before running: {path}")


def show_figure(name, figure):
    figure.update_layout(template="plotly_white", font=dict(size=13), legend_title_text="Estimator")
    FIGURES[name] = figure
    display(HTML(figure.to_html(full_html=False, include_plotlyjs="cdn", div_id=f"vad-{name}")))

## 1. Context contrasts: 20 pairs, three estimators

The summaries use the same pairs with complete VAD for all three estimators. Raw values and deltas use the common 1–5 scale; normalized differences divide that amplitude by four. All individual and missing pair results remain in the exported tables. Positive context does not imply larger arousal or dominance by design.

In [ ]:
context_scored, context_pairs, context_summary, context_manifest = (
    run_three_model_context_comparison(
        CONTEXT_INPUT,
        MEMOLON_PATH,
        NRC_PATH,
        CONTEXT_OUTPUT_DIR,
        bert_cache_path=BERT_CACHE_PATH,
        historical_baseline_path=HISTORICAL_BASELINE,
    )
)
print(
    f"Pairs: {context_manifest['pairs']:.0f}; "
    f"jointly valid: {context_manifest['joint_valid_pairs']}"
)
print(f"Current BERT revision: {context_manifest['bert_baseline']['revision']}")
display(
    context_summary[
        [
            "model",
            "dimension",
            "valid_pairs",
            "mean_delta",
            "mean_abs_delta",
            "normalized_mean_abs_delta",
            "positive_delta_count",
        ]
    ]
)
display(
    pd.DataFrame(
        [
            {"model": MODEL_LABELS[key], "token_coverage": value}
            for key, value in context_manifest["coverage"].items()
        ]
    )
)

In [ ]:
for metric, title in [
    ("normalized_mean_abs_delta", "Context contrast magnitude on the common 0-1 scale"),
    ("mean_delta", "Signed context contrast: positive minus negative, common 1-5 scale"),
]:
    figure = px.bar(
        context_summary,
        x="dimension",
        y=metric,
        color="model",
        barmode="group",
        color_discrete_map=COLORS,
        title=title,
        hover_data=["valid_pairs", "positive_delta_count"],
    )
    show_figure(f"context_{metric}", figure)

long = context_pairs.melt(
    id_vars=["pair_id", "model", "focal_event"],
    value_vars=[f"normalized_delta_{d}" for d in ["valence", "arousal", "dominance"]],
    var_name="dimension",
    value_name="signed_delta",
)
figure = px.line(
    long,
    x="pair_id",
    y="signed_delta",
    color="model",
    facet_col="dimension",
    markers=True,
    color_discrete_map=COLORS,
    hover_data=["focal_event"],
    title="Individual context pair contrasts, normalized signed differences",
)
show_figure("context_individual_pairs", figure)

## 2. Inspect the same context pair

The two texts can be read next to the three sets of VAD scores. BERT is a comparator, not an independent human reference.

In [ ]:
SELECTED_PAIR_ID = context_scored.pair_id.iloc[0]
pair_rows = context_scored.loc[context_scored.pair_id == SELECTED_PAIR_ID]
display(
    pair_rows[
        [
            "model",
            "context_polarity",
            "vad_valence",
            "vad_arousal",
            "vad_dominance",
            "token_coverage",
        ]
    ]
)
for row in pair_rows.loc[pair_rows.model == MODEL_LABELS["bert"]].itertuples():
    print(f"{row.context_polarity.upper()}: {row.article_text}")
    print()
display(Markdown((CONTEXT_OUTPUT_DIR / "findings.md").read_text(encoding="utf-8")))

## 3. Saved simulation chains

All three estimators use the same saved texts and original/input references. Hypothetical STDI preserves the historical non-VAD contribution and substitutes only the VAD contribution after validating the historical formula and saved BERT drift. Scores missing for any estimator are excluded from joint summaries, with no neutral imputation.

The chain filter defaults to SSSS, CCCC, PPPP, DDDD, CCPP, and PPCC. Change it to a different tuple or `None` to expand the scope.

In [ ]:
steps, summary, manifest = run_three_model_simulation_comparison(
    RUNS_DIR,
    MEMOLON_PATH,
    NRC_PATH,
    OUTPUT_DIR,
    bert_cache_path=BERT_CACHE_PATH,
    chain_codes=CHAIN_CODES,
)
print(
    f"Executions: {manifest['runs']}; news items: {manifest['news_items']}; "
    f"steps: {manifest['steps']}"
)
print(
    f"Unique texts: {manifest['unique_texts_scored']}; "
    f"BERT truncated inputs: {manifest['bert_baseline']['truncated_texts']}"
)
assert manifest["bert_baseline"]["revision"] == context_manifest["bert_baseline"]["revision"]
display(pd.read_csv(OUTPUT_DIR / "run_inventory.csv"))
display(
    steps.groupby(["chain_code", "formula_check_vs_original", "nrc_formula_check_vs_original"])
    .size()
    .rename("steps")
    .reset_index()
)
display(Markdown((OUTPUT_DIR / "findings.md").read_text(encoding="utf-8")))

## 4. VAD drift and hypothetical STDI by chain and step

Every point uses the same available observations for all three estimators. Original-relative drift measures distance from the initial text; incremental drift measures change relative to the input actually received.

In [ ]:
MEAN_LABELS = {f"{model}_mean": label for model, label in MODEL_LABELS.items()}
for metric, relation, title in [
    ("vad_drift", "vs_original", "VAD drift relative to the original"),
    ("vad_drift", "incremental", "Incremental VAD drift"),
    ("stdi", "vs_original", "Hypothetical STDI relative to the original"),
]:
    selected = summary.loc[(summary.metric == metric) & (summary.relation == relation)]
    long = selected.melt(
        id_vars=["run_id", "chain_code", "step_index", "paired_steps"],
        value_vars=list(MEAN_LABELS),
        var_name="model",
        value_name="mean_value",
    )
    long["model"] = long.model.map(MEAN_LABELS)
    figure = px.line(
        long,
        x="step_index",
        y="mean_value",
        color="model",
        markers=True,
        line_group="run_id",
        facet_col="chain_code",
        facet_col_wrap=3,
        color_discrete_map=COLORS,
        hover_data=["run_id", "paired_steps"],
        title=title,
    )
    figure.update_xaxes(dtick=1)
    figure.update_layout(height=560)
    show_figure(f"{metric}_{relation}", figure)

## 5. Each lexical estimator relative to current BERT

Heatmaps use one shared symmetric color range for both lexical estimators within each metric. Positive values mean a larger result than BERT. Numerical agreement does not establish accuracy.

In [ ]:
for metric in ("vad_drift", "stdi"):
    selected = summary.loc[(summary.metric == metric) & (summary.relation == "vs_original")]
    bound = max(float(selected[["nrc_change_mean", "memolon_change_mean"]].abs().max().max()), 1e-6)
    for model in ("nrc", "memolon"):
        matrix = selected.pivot(
            index="chain_code", columns="step_index", values=f"{model}_change_mean"
        )
        figure = px.imshow(
            matrix,
            text_auto=".4f",
            aspect="auto",
            color_continuous_scale="RdBu_r",
            range_color=[-bound, bound],
            title=f"{metric}: {MODEL_LABELS[model]} minus current BERT",
        )
        show_figure(f"change_{model}_{metric}", figure)

## 6. Distributions and dimensions

Steps from the same news item and across chains are dependent; these plots are descriptive. All three distributions use the same available steps.

In [ ]:
drift_columns = {f"{m}_vad_drift_vs_original": label for m, label in MODEL_LABELS.items()}
paired = steps.dropna(subset=list(drift_columns))
distribution = paired.melt(
    id_vars=["chain_code", "news_id", "step_index"],
    value_vars=list(drift_columns),
    var_name="model",
    value_name="drift",
)
distribution["model"] = distribution.model.map(drift_columns)
figure = px.box(
    distribution,
    x="chain_code",
    y="drift",
    color="model",
    color_discrete_map=COLORS,
    title="Original-relative VAD drift: same steps for all three estimators",
)
show_figure("drift_distribution", figure)

SELECTED_CHAIN_CODE = "SSSS"
selected = summary.loc[
    (summary.chain_code == SELECTED_CHAIN_CODE)
    & (summary.relation == "vs_original")
    & summary.metric.isin(["valence_drift", "arousal_drift", "dominance_drift"])
]
long = selected.melt(
    id_vars=["run_id", "metric", "step_index", "paired_steps"],
    value_vars=list(MEAN_LABELS),
    var_name="model",
    value_name="drift",
)
long["model"] = long.model.map(MEAN_LABELS)
figure = px.line(
    long,
    x="step_index",
    y="drift",
    color="model",
    facet_col="metric",
    markers=True,
    line_group="run_id",
    color_discrete_map=COLORS,
    title=f"Normalized drift by dimension: {SELECTED_CHAIN_CODE}",
)
show_figure("dimension_comparison", figure)

## 7. Follow one news item with all three estimators

Step 0 is the original news item. Scores and drift are different quantities; higher scores do not automatically imply greater drift. Set `SELECTED_NEWS_ID` to inspect another item.

In [ ]:
pool = steps.loc[steps.chain_code == SELECTED_CHAIN_CODE]
SELECTED_RUN_ID = pool.run_id.iloc[0]
pool = pool.loc[pool.run_id == SELECTED_RUN_ID]
display(pool[["news_id", "metadata_title"]].drop_duplicates())
SELECTED_NEWS_ID = pool.news_id.iloc[0]
example = pool.loc[pool.news_id == SELECTED_NEWS_ID].sort_values("step_index")
figure = make_subplots(rows=1, cols=3, subplot_titles=["Valence", "Arousal", "Dominance"])
for column, dim in enumerate(["valence", "arousal", "dominance"], start=1):
    for model, label in MODEL_LABELS.items():
        values = [example.iloc[0][f"{model}_original_{dim}_normalized"]]
        values += example[f"{model}_rewritten_{dim}_normalized"].tolist()
        figure.add_trace(
            go.Scatter(
                x=[0] + example.step_index.tolist(),
                y=values,
                name=label,
                legendgroup=model,
                showlegend=column == 1,
                mode="lines+markers",
                connectgaps=False,
                line=dict(color=COLORS[label]),
            ),
            row=1,
            col=column,
        )
figure.update_yaxes(range=[0, 1], title_text="Normalized score")
figure.update_xaxes(dtick=1, title_text="Step (0 = original)")
show_figure("individual_trajectory", figure)
display(
    example[
        ["step_index", "node_label"]
        + [f"{m}_{metric}_vs_original" for metric in ["vad_drift", "stdi"] for m in MODEL_LABELS]
    ]
)
print("ORIGINAL:")
print(example.iloc[0].original_text)
for row in example.itertuples():
    print()
    print(f"STEP {row.step_index} | {row.node_label}")
    print(row.rewritten_text)

## 8. Coverage, exports, and interpretation

Lexical coverage is reported separately from affective validity. BERT truncates at 512 WordPiece tokens, whereas the lexicons process the full text. Cumulative STDI can exceed one and is not original-relative drift. These exploratory comparisons do not validate Portuguese, factual veracity, belief, exposure, or sharing.

In [ ]:
coverage = steps.melt(
    id_vars=["chain_code", "news_id", "step_index"],
    value_vars=["nrc_rewritten_token_coverage", "memolon_rewritten_token_coverage"],
    var_name="model",
    value_name="coverage",
)
coverage["model"] = coverage.model.map(
    {
        "nrc_rewritten_token_coverage": MODEL_LABELS["nrc"],
        "memolon_rewritten_token_coverage": MODEL_LABELS["memolon"],
    }
)
figure = px.histogram(
    coverage,
    x="coverage",
    color="model",
    barmode="overlay",
    opacity=0.6,
    nbins=30,
    color_discrete_map=COLORS,
    title="Lexical coverage of rewritten texts",
)
figure.update_xaxes(range=[0, 1])
show_figure("lexical_coverage", figure)

sections = [
    figure.to_html(full_html=False, include_plotlyjs=(index == 0), div_id=f"vad-{name}")
    for index, (name, figure) in enumerate(FIGURES.items())
]
html = "<!doctype html><html lang='en'><head><meta charset='utf-8'>"
html += "<meta name='viewport' content='width=device-width, initial-scale=1'>"
html += "<title>Three-model VAD comparison</title></head>"
html += "<body style='max-width:1400px;margin:30px auto;font-family:sans-serif'>"
html += "<h1>Current BERT, NRC v2.1, and MEmoLon</h1>"
html += "<p>20 synthetic English context pairs and six saved simulation chains. "
html += "Shared theoretical scales; BERT is a comparator, not a human gold standard. "
html += "No human validation or Portuguese validation.</p>"
html += "".join(sections) + "</body></html>"
dashboard_path = OUTPUT_DIR / "comparison_dashboard.html"
temporary_dashboard = dashboard_path.with_suffix(".html.tmp")
temporary_dashboard.write_text(html, encoding="utf-8")
temporary_dashboard.replace(dashboard_path)
print(f"Simulation results: {OUTPUT_DIR}")
print(f"Context results: {CONTEXT_OUTPUT_DIR}")
print(f"Offline HTML export ({len(FIGURES)} charts): {dashboard_path}")

## Primary resource references

- [NRC VAD v2 paper](https://arxiv.org/abs/2503.23547) and [resource page](https://saifmohammad.com/WebPages/nrc-vad.html).
- [MEmoLon paper](https://aclanthology.org/2020.acl-main.112/) and [official dataset](https://zenodo.org/records/3756607).
- [Current BERT model card](https://huggingface.co/RobroKools/vad-bert). Its exact revision is recorded in both manifests.